# Proyek Akhir: Telco Customer Churn

**Nama:** [Nama Anda]  
**Email:** [Email Anda]  
**Dataset:** [Telco Customer Churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)  

---

## 1. Data Loading
Memuat dataset Telco Customer Churn dari file CSV.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

## 2. Exploratory Data Analysis (EDA)
Memahami distribusi data, tipe data, dan statistik deskriptif.

In [ ]:
# Info tipe data
df.info()

In [ ]:
# Statistik deskriptif
df.describe(include='all')

In [ ]:
# Distribusi target variable
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

churn_counts = df['Churn'].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%', colors=['#4CAF50', '#F44336'])
axes[0].set_title('Distribusi Churn')

sns.countplot(x='Churn', data=df, palette=['#4CAF50', '#F44336'], ax=axes[1])
axes[1].set_title('Count Churn vs No Churn')

plt.tight_layout()
plt.show()

print(churn_counts)
print(f'Churn rate: {churn_counts["Yes"]/len(df)*100:.2f}%')

In [ ]:
# Distribusi fitur numerik
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, col in enumerate(num_cols):
    df_temp = df.copy()
    df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')
    df_temp[col].hist(bins=30, ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(f'Distribusi {col}')
    axes[i].set_xlabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Distribusi fitur kategorikal
cat_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'Contract',
            'PaymentMethod', 'InternetService', 'PhoneService']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.ravel()

for i, col in enumerate(cat_cols):
    df[col].value_counts().plot(kind='bar', ax=axes[i], color='steelblue')
    axes[i].set_title(col)
    axes[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 3. Missing Value Analysis
Mengidentifikasi dan menangani nilai yang hilang.

In [ ]:
# Cek missing values awal
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing, 'Persen (%)': missing_pct})
print('Missing values per kolom:')
print(missing_df[missing_df['Missing'] > 0])

In [ ]:
# TotalCharges mengandung spasi yang harus dikonversi ke NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

missing_after = df.isnull().sum()
print('Missing values setelah konversi TotalCharges:')
print(missing_after[missing_after > 0])

In [ ]:
# Imputasi TotalCharges dengan median
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

print('Missing values setelah imputasi:')
print(df.isnull().sum().sum(), 'total missing values')

## 4. Duplicate Analysis
Memeriksa dan menghapus baris duplikat.

In [ ]:
dup_count = df.duplicated().sum()
print(f'Jumlah baris duplikat: {dup_count}')

if dup_count > 0:
    df.drop_duplicates(inplace=True)
    print(f'Duplikat dihapus. Shape baru: {df.shape}')
else:
    print('Tidak ada duplikat. Dataset bersih.')

print(f'Shape akhir: {df.shape}')

## 5. Outlier Analysis
Mendeteksi dan menangani outlier pada fitur numerik menggunakan metode IQR.

In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(f'Boxplot {col}')
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
def count_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = series[(series < lower) | (series > upper)]
    return len(outliers), lower, upper

for col in num_cols:
    count, low, high = count_outliers_iqr(df[col])
    print(f'{col}: {count} outlier | batas bawah={low:.2f}, batas atas={high:.2f}')

In [ ]:
# Capping outlier dengan batas IQR (Winsorization)
def cap_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower, upper=upper)
    return df

for col in num_cols:
    df = cap_outliers_iqr(df, col)

print('Outlier setelah capping:')
for col in num_cols:
    count, _, _ = count_outliers_iqr(df[col])
    print(f'  {col}: {count} outlier')

## 6. Feature Engineering
Membuat fitur baru yang relevan dari data yang sudah ada.

In [ ]:
# Fitur 1: Avg Monthly Charge per tenure
df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'] + 1)

# Fitur 2: Jumlah layanan yang digunakan
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

def count_services(row):
    count = 0
    for col in service_cols:
        val = str(row[col]).lower()
        if val not in ['no', 'no internet service', 'no phone service']:
            count += 1
    return count

df['NumServices'] = df.apply(count_services, axis=1)

# Fitur 3: HasFamily (Partner atau Dependents)
df['HasFamily'] = ((df['Partner'] == 'Yes') | (df['Dependents'] == 'Yes')).astype(int)

# Fitur 4: Tenure group
df['TenureGroup'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72],
                            labels=['0-12m', '13-24m', '25-48m', '49-72m'], right=True)

print('Fitur baru:')
print(df[['AvgMonthlyCharge', 'NumServices', 'HasFamily', 'TenureGroup']].head(10))

## 7. Encoding
Mengubah fitur kategorikal menjadi numerik.

In [ ]:
df_encoded = df.copy()

# Drop kolom ID
df_encoded.drop(columns=['customerID'], inplace=True)

# Encode target: Yes=1, No=0
df_encoded['Churn'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})

# Encode binary cols
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
le = LabelEncoder()
for col in binary_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

# One-Hot Encoding untuk kolom multi-kategori
ohe_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
            'Contract', 'PaymentMethod', 'TenureGroup']

df_encoded = pd.get_dummies(df_encoded, columns=ohe_cols, drop_first=True)

# Pastikan semua kolom bool dikonversi ke int
bool_cols = df_encoded.select_dtypes(include='bool').columns
df_encoded[bool_cols] = df_encoded[bool_cols].astype(int)

print('Shape setelah encoding:', df_encoded.shape)
df_encoded.head()

## 8. Scaling
Menormalisasi fitur numerik menggunakan StandardScaler.

In [ ]:
scale_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'AvgMonthlyCharge', 'NumServices']

scaler = StandardScaler()
df_encoded[scale_cols] = scaler.fit_transform(df_encoded[scale_cols])

print('Statistik setelah scaling:')
df_encoded[scale_cols].describe().round(4)

## 9. Train Test Split
Membagi dataset menjadi data latih (80%) dan data uji (20%).

In [ ]:
X = df_encoded.drop(columns=['Churn'])
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Total data  : {len(df_encoded)}')
print(f'X_train     : {X_train.shape}')
print(f'X_test      : {X_test.shape}')
print(f'y_train dist:\n{y_train.value_counts(normalize=True).round(3)}')
print(f'y_test dist:\n{y_test.value_counts(normalize=True).round(3)}')

## 10. Menyimpan Dataset Hasil Preprocessing
Menyimpan dataset yang telah diproses ke file CSV untuk digunakan pada tahap pelatihan model.

In [ ]:
# Simpan dataset lengkap
df_encoded.to_csv('telco_churn_preprocessed.csv', index=False)

# Simpan split terpisah
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print('File yang disimpan:')
print('  telco_churn_preprocessed.csv  -> Dataset lengkap setelah preprocessing')
print('  X_train.csv / X_test.csv      -> Fitur train & test')
print('  y_train.csv / y_test.csv      -> Label train & test')
print(f'\nShape final dataset: {df_encoded.shape}')
print(f'Jumlah fitur: {X_train.shape[1]}')